# DVCA Cross-Slice Workflow

This notebook is organized as a DVCA analysis pipeline: environment bootstrap, slice assembly, representation learning, domain evaluation, cross-slice alignment, visualization, and export.


## 1. Environment Setup

This section initializes the DVCA runtime, resolves the project path, configures the R dependency used by `mclust`, and defines the shared analysis constants for the whole notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import site
import sys
from pathlib import Path

import anndata as ad
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.linalg
import seaborn as sns
import torch
from IPython.display import display
from sklearn.metrics import (
    adjusted_mutual_info_score as ami_score,
    adjusted_rand_score as ari_score,
    completeness_score as com_score,
    homogeneity_score as hom_score,
    normalized_mutual_info_score as nmi_score,
)
from sklearn.metrics.pairwise import euclidean_distances


def resolve_project_root(start_dir: Path) -> Path:
    candidates = [
        start_dir,
        *start_dir.parents,
        Path(r"D:/fuxian/DVCAlign"),
        Path(r"D:/fuxian"),
    ]
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "setup.py").exists() and (candidate / "DVCAlign" / "training.py").exists():
            return candidate
        nested = candidate / "DVCAlign"
        if (nested / "setup.py").exists() and (nested / "DVCAlign" / "training.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the DVCAlign project root.")


def configure_r_home() -> None:
    if os.environ.get("R_HOME"):
        return
    candidates = [
        Path(r"C:/Program Files/R/R-4.4.1"),
        Path(r"C:/Program Files/R/R-4.4.0"),
        Path(r"C:/Program Files/R/R-4.3.3"),
        Path(r"C:/Program Files/R/R-4.3.2"),
        Path("/usr/lib/R"),
    ]
    for candidate in candidates:
        if candidate.exists():
            os.environ["R_HOME"] = str(candidate)
            return


NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = resolve_project_root(NOTEBOOK_DIR)
DATA_ROOT = PROJECT_ROOT / "DVCAlign" / "Data"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

configure_r_home()
for package_dir in site.getsitepackages():
    rpy2_dir = Path(package_dir) / "rpy2"
    if rpy2_dir.exists() and package_dir not in sys.path:
        sys.path.append(package_dir)

import DVCAlign
import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri

RANDOM_SEED = 666
np.random.seed(RANDOM_SEED)
robjects.r(f"set.seed({RANDOM_SEED})")
robjects.r('RNGkind("Mersenne-Twister")')

DEVICE = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
SECTION_IDS = ["151673", "151674", "151675", "151676"]
SLICE_TAG = f"{SECTION_IDS[0]}_{SECTION_IDS[-1]}"
EMBEDDING_KEY = "DVCAlign"
LABEL_KEY = "Ground Truth"
SLICE_KEY = "slice_name"
BATCH_KEY = "batch_name"
CLUSTER_KEY = "mclust"
DOMAIN_KEY = "dvca_domain"
N_CLUSTERS = 7
SPATIAL_RADIUS = 150
N_TOP_GENES = 5000

sns.set(style="white", font_scale=1.1)
plt.rcParams["figure.dpi"] = 160
plt.rcParams["savefig.dpi"] = 300

print("Project root:", PROJECT_ROOT)
print("Notebook dir:", NOTEBOOK_DIR)
print("Data root:", DATA_ROOT)
print("Device:", DEVICE)
print("Sections:", SECTION_IDS)


Project root: /workspace/DVCAlign
Notebook dir: /workspace/DVCAlign/DVCAlign/Notebooks/DLPFC12
Data root: /workspace/DVCAlign/DVCAlign/Data
Device: cuda:2
Sections: ['151673', '151674', '151675', '151676']


## 2. Slice Preparation and Graph Assembly

Here we load each Visium slice, attach the manual labels, construct the within-slice spatial graph, apply gene filtering and normalization, and then assemble all slices into a single graph object for cross-slice training.

In [2]:
def prepare_dvca_slice(section_id, data_root=DATA_ROOT, rad_cutoff=SPATIAL_RADIUS, n_top_genes=N_TOP_GENES):
    input_dir = data_root / section_id
    adata = sc.read_visium(
        path=input_dir,
        count_file=f"{section_id}_filtered_feature_bc_matrix.h5",
        load_images=True,
    )
    adata.var_names_make_unique(join="++")

    ann_df = pd.read_csv(input_dir / f"{section_id}_truth.txt", sep="\t", header=None, index_col=0)
    ann_df.columns = [LABEL_KEY]
    ann_df[ann_df.isna()] = "unknown"
    adata.obs[LABEL_KEY] = ann_df.loc[adata.obs_names, LABEL_KEY].astype("category")
    adata.obs_names = [f"{name}_{section_id}" for name in adata.obs_names]

    DVCAlign.Cal_Spatial_Net(adata, rad_cutoff=rad_cutoff)
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", n_top_genes=n_top_genes)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata = adata[:, adata.var["highly_variable"]].copy()
    return adata


def assemble_dvca_graph(slice_adatas, section_ids):
    adata_concat = ad.concat(slice_adatas, label=SLICE_KEY, keys=section_ids)
    adata_concat.obs[LABEL_KEY] = adata_concat.obs[LABEL_KEY].astype("category")
    adata_concat.obs[BATCH_KEY] = adata_concat.obs[SLICE_KEY].astype("category")

    adj_concat = np.asarray(slice_adatas[0].uns["adj"].todense())
    for batch_id in range(1, len(section_ids)):
        adj_concat = scipy.linalg.block_diag(adj_concat, np.asarray(slice_adatas[batch_id].uns["adj"].todense()))
    adata_concat.uns["edgeList"] = np.nonzero(adj_concat)
    return adata_concat


def ensure_projection_map(adata, use_rep=EMBEDDING_KEY, n_neighbors=15, min_dist=0.35):
    if "X_umap" not in adata.obsm:
        sc.pp.neighbors(adata, use_rep=use_rep, n_neighbors=n_neighbors, random_state=RANDOM_SEED)
        sc.tl.umap(adata, min_dist=min_dist, random_state=RANDOM_SEED)
    return adata


def normalize_categories(series):
    return series.astype(str).astype("category")


def take_slice_view(adata, slice_id, batch_col=BATCH_KEY):
    return adata[adata.obs[batch_col].astype(str) == str(slice_id)].copy()


def summarize_domain_metrics(adata, truth_col=LABEL_KEY, pred_col=DOMAIN_KEY):
    valid = adata.obs[[truth_col, pred_col]].dropna().copy()
    valid = valid[valid[truth_col].astype(str) != "unknown"]
    y_true = valid[truth_col].astype(str)
    y_pred = valid[pred_col].astype(str)
    return {
        "ARI": ari_score(y_true, y_pred),
        "NMI": nmi_score(y_true, y_pred),
        "AMI": ami_score(y_true, y_pred),
        "HOM": hom_score(y_true, y_pred),
        "COM": com_score(y_true, y_pred),
    }


def compute_slice_alignment_scores(adata, embedding_key=EMBEDDING_KEY, batch_key=BATCH_KEY, label_key=LABEL_KEY):
    adata_eval = adata[adata.obs[label_key].astype(str) != "unknown"].copy()
    slice_ids = [str(x) for x in adata_eval.obs[batch_key].cat.categories]
    slice_pairs = list(zip(slice_ids[:-1], slice_ids[1:]))

    rows = []
    batch_values = adata_eval.obs[batch_key].astype(str)
    for src_slice, tgt_slice in slice_pairs:
        src = adata_eval[batch_values == src_slice].copy()
        tgt = adata_eval[batch_values == tgt_slice].copy()
        z_src = np.asarray(src.obsm[embedding_key])
        z_tgt = np.asarray(tgt.obsm[embedding_key])
        dist = euclidean_distances(z_src, z_tgt)
        nn_idx = dist.argmin(axis=1)
        src_labels = src.obs[label_key].astype(str).to_numpy()
        tgt_labels = tgt.obs[label_key].astype(str).to_numpy()
        matched_labels = tgt_labels[nn_idx]
        paa = float((src_labels == matched_labels).mean())
        unique_targets = len(np.unique(nn_idx))
        smr = float(len(nn_idx) / max(unique_targets, 1))
        rows.append({
            "source_slice": src_slice,
            "target_slice": tgt_slice,
            "n_anchor_spots": int(len(nn_idx)),
            "n_unique_target_spots": int(unique_targets),
            "PAA": paa,
            "SMR": smr,
        })
    return pd.DataFrame(rows)


slice_adatas = [prepare_dvca_slice(section_id) for section_id in SECTION_IDS]
adata_concat = assemble_dvca_graph(slice_adatas, SECTION_IDS)
print("adata_concat.shape:", adata_concat.shape)


------Calculating spatial graph...
The graph contains 21124 edges, 3639 cells.
5.8049 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 21258 edges, 3673 cells.
5.7876 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 20762 edges, 3592 cells.
5.7801 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 20052 edges, 3460 cells.
5.7954 neighbors per cell on average.
adata_concat.shape: (14364, 1125)


## 3. Training Configuration

The following cell collects the core optimization settings for DVCA, including the hidden dimensions, learning rate, regularization, neighborhood size, and random seed.

In [3]:
fit_config = dict(
    hidden_dims=[512, 32],
    n_epochs=1000,
    lr=0.001,
    gradient_clipping=5.0,
    weight_decay=0.001,
    margin=1.0,
    knn_neigh=20,
    verbose=True,
    random_seed=RANDOM_SEED,
    device=DEVICE,
)
fit_config


{'hidden_dims': [512, 32],
 'n_epochs': 1000,
 'lr': 0.001,
 'gradient_clipping': 5.0,
 'weight_decay': 0.0001,
 'margin': 1.0,
 'knn_neigh': 20,
 'verbose': True,
 'random_seed': 666,
 'device': device(type='cuda', index=2)}

## 4. Representation Learning

This is the main training step. The DVCA model is fitted on the concatenated graph and writes the learned embedding into the AnnData object for downstream clustering and alignment analysis.

In [4]:
%%time
adata_concat = DVCAlign.train_DVCAlign(adata_concat, **fit_config)


DVCAlignModel(
  (conv1): GATConv(1125, 512, heads=1)
  (conv2): GATConv(512, 32, heads=1)
  (conv3): GATConv(32, 512, heads=1)
  (conv4): GATConv(512, 1125, heads=1)
  (residual): Linear(in_features=1125, out_features=32, bias=True)
  (view_gate): Linear(in_features=64, out_features=1, bias=True)
)
Use dual-view RC loss: lam_re=1.0, lam_rc=1.0, lam_dec=0.05, triplet_warmup_epochs=200, triplet_weight_max=0.7
Use spatial graph + local-expression graph dual views: aux_candidate_k=40, aux_expr_k=20, aux_pca_dim=30, spatial_fusion_weight=0.5
Triplet positive strategy: pick the closest spot among MNN candidates.
Use adaptive dual-view fusion: True; use confidence-aware triplet: True; use chain consistency: True
Pretrain DVCAlign encoder...


100%|██████████| 500/500 [00:57<00:00,  8.62it/s]


Train DVCAlign...


  0%|          | 0/500 [00:00<?, ?it/s]

Update spot triplets at epoch 500


...

Update spot triplets at epoch 800


 80%|████████  | 400/500 [01:07<00:11,  8.37it/s]

Update spot triplets at epoch 900


100%|██████████| 500/500 [01:24<00:00,  5.92it/s]

CPU times: user 3min 16s, sys: 52.6 s, total: 4min 9s
Wall time: 2min 24s


## 5. Domain Identification

After the embedding is learned, `mclust` is used to partition the latent space into spatial domains. A DVCA-specific domain label is then created for cleaner downstream reporting.

In [ ]:
adata_concat = DVCAlign.mclust_R(adata_concat, num_cluster=N_CLUSTERS, used_obsm=EMBEDDING_KEY)
adata_concat.obs[DOMAIN_KEY] = adata_concat.obs[CLUSTER_KEY].astype(str).astype('category')
adata_eval = adata_concat[adata_concat.obs[LABEL_KEY] != 'unknown'].copy()

print(f'{DOMAIN_KEY}, ARI = {ari_score(adata_eval.obs[LABEL_KEY], adata_eval.obs[DOMAIN_KEY]):0.3f}')
print(f'{DOMAIN_KEY}, NMI = {nmi_score(adata_eval.obs[LABEL_KEY], adata_eval.obs[DOMAIN_KEY]):0.3f}')


R[write to console]:                    __           __ 
   ____ ___  _____/ /_  _______/ /_
  / __ `__ \/ ___/ / / / / ___/ __/
 / / / / / / /__/ / /_/ (__  ) /_  
/_/ /_/ /_/\___/_/\__,_/____/\__/   version 6.1.1
Type 'citation("mclust")' for citing this R package in publications.



fitting ...
  |======================================================================| 100%
